<a href="https://colab.research.google.com/github/JordanTerwilliger/Intro-to-Deep-Learning/blob/main/HW2/HW2_LanguageProcessing_RNN_Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [177]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import requests

from torch import nn
from torch import functional as F
from torch import optim

!pip install torchinfo

import matplotlib.pyplot as plt


from torchinfo import summary

In [178]:
torch.manual_seed(1)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


#hyperparameters
lr = 0.01
epochs = 300
input_lengths = [10,20,30]
hidden_size = 100

device

device(type='cuda', index=0)

In [ ]:
text_sequence = "Next character prediction is a fundamental task in the field of natural language processing (NLP) that involves predicting the next character in a sequence of text based on the characters that precede it. This task is essential for various applications, including text auto-completion, spell checking, and even in the development of sophisticated AI models capable of generating human-like text."

In [180]:
# We need to convert this text into a list of sorted indices for

sorted_text = list(sorted(set(text_sequence)))
print(sorted_text)

ix_to_char = {i: ch for i,ch in enumerate(sorted_text)}
print(ix_to_char)

char_to_ix = {ch: i for i, ch in enumerate(sorted_text)}

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}


In [181]:
#Create Model
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super(RNN, self).__init__()
    self.embedding = nn.Embedding(input_size, hidden_size)
    self.rnn = nn.RNN(input_size = hidden_size, hidden_size=hidden_size, batch_first = True)
    self.fc1 = nn.Linear(in_features = hidden_size, out_features = output_size)


  def forward(self, x):
    embedded = self.embedding(x)
    output,_ = self.rnn(embedded)
    output = self.fc1(output[:, -1, :])
    return output

In [182]:
model = RNN(len(sorted_text), hidden_size, len(sorted_text)).to(device)
criterion = nn.CrossEntropyLoss()
optim = optim.SGD(model.parameters(), lr = lr)

In [183]:
#Splitting into train and test datasets
from sklearn.model_selection import train_test_split
def preprocess_text(text, sequence_length):
    # Preparing the dataset
    max_length = sequence_length  # Maximum length of input sequences
    X = []
    y = []
    for i in range(len(text) - max_length):
        sequence = text[i:i + max_length]
        label = text[i + max_length]
        X.append([char_to_ix[char] for char in sequence])
        y.append(char_to_ix[label])

    X = np.array(X)
    y = np.array(y)

    # Splitting the dataset into training and validation sets
    return train_test_split(X, y, test_size=0.2, random_state=1)

In [184]:
def createPlot(sequence_length):
  plt.figure(figsize=(12, 5))

  # Plot Loss
  plt.subplot(1, 2, 1)
  plt.plot(train_loss_list, label='Train Loss')
  plt.plot(val_loss_list, label='Val Loss')
  plt.title(f'Loss Curves Sequence Length:{sequence_length}')
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.legend()

  # Plot Accuracy
  plt.subplot(1, 2, 2)
  plt.plot(val_accuracy_list, label='Val Accuracy')
  plt.title(f'Validation Accuracy Sequence Length:{sequence_length}')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy (%)')
  plt.legend()

  plt.show()

In [185]:


##Data Preprocessing and converting to tensor
for i, input_length in enumerate(input_lengths):
  X_train, X_val, y_train, y_val = preprocess_text(text_sequence, input_length)

  X_train = torch.tensor(X_train, dtype=torch.long).to(device)
  y_train = torch.tensor(y_train, dtype=torch.long).to(device)
  X_val = torch.tensor(X_val, dtype=torch.long).to(device)
  y_val = torch.tensor(y_val, dtype=torch.long).to(device)




  ###This is where training begins
  #Lists for storing loss and validation values
  train_loss_list = []
  val_loss_list = []
  val_accuracy_list = []


  #Create a new training loop for each input_length
  for epoch in range(epochs):
    model.train()

    optim.zero_grad()
    y_pred = model(X_train)  # Forward pass
    loss = criterion(y_pred, y_train)  # Compute loss
    loss.backward()  # Backward pass
    optim.step()  # Update model parameters

    train_loss_list.append(loss.item()) #Take this epoch's training loss and add it
                                        #to the training loss list (of all epochs)


    #Here is where we evaluate the model on the current epoch
    model.eval()
    with torch.no_grad():
      val_output = model(X_val) # Take test dataset and run it through this epoch's model

      val_loss = criterion(val_output, y_val) #Find the loss

      _, predicted = torch.max(val_output, 1) #Here we find what the output was (what letter)

      val_accuracy = (predicted == y_val).float().mean() #Here we take each answer from out model,
                                                         #compare it to the ground truth, and find how accurate we are

      val_loss_list.append(val_loss.item())
      val_accuracy_list.append(val_accuracy.item())

      if(epoch % 100) == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}, Val Accuracy: {val_accuracy.item():.4f}, Val Loss: {val_loss.item():.4f}')

  createPlot(sequence_length=i)



OutOfMemoryError: CUDA out of memory. Tried to allocate 51.87 GiB. GPU 0 has a total capacity of 39.49 GiB of which 28.19 GiB is free. Including non-PyTorch memory, this process has 11.29 GiB memory in use. Of the allocated memory 10.75 GiB is allocated by PyTorch, and 44.20 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
summary(model, input_size = (1,20), dtypes=[torch.long])